In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

#### **Data Ingestion**

In [2]:
from langchain_community.document_loaders import PyPDFLoader

In [3]:
loaders = [
    PyPDFLoader("../lost-in-middle-paper.pdf"),
    PyPDFLoader("../Retrieval-Augmented-Generation.pdf"),
]

docs = []
for loader in loaders:
    docs.extend(loader.load())

In [5]:
len(docs)

37

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [9]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.0
)

#### **Retrival Pipeline**

In [11]:
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma

In [12]:
vectorstore = Chroma(collection_name="full_documents", embedding_function=embeddings)

In [13]:

store = InMemoryStore()

In [14]:

from langchain.retrievers import ParentDocumentRetriever

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

In [15]:

retriever.add_documents(docs, ids=None)

In [17]:

len(list(store.yield_keys()))

37

In [20]:
retrieved_docs = retriever.invoke("How might alternative positional encoding methods or attention mechanisms mitigate this degradation in the middle of long contexts?")

In [22]:
retrieved_docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-11-22T01:09:40+00:00', 'author': '', 'keywords': '', 'moddate': '2023-11-22T01:09:40+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../lost-in-middle-paper.pdf', 'total_pages': 18, 'page': 6, 'page_label': '7'}, page_content='1st 25th 50th 75th\nPosition of Key to Retrieve\n40\n50\n60\n70\n80\n90\n100Accuracy\n75 Key-Value Pairs (~4K tokens)\n1st 35th 70th 105th 140th\nPosition of Key to Retrieve\n40\n50\n60\n70\n80\n90\n100Accuracy\n140 Key-Value Pairs (~8K tokens)\n1st 50th 100th 150th 200th 250th 300th\nPosition of Key to Retrieve\n40\n50\n60\n70\n80\n90\n100Accuracy\n300 Key-Value Pairs (~16K tokens)\nclaude-1.3 claude-1.3-100k gpt-3.5-turbo-0613 gpt-3.5-turbo-16k-0613 mpt-30b-instruct longchat-13b-16k\nFigure 7: The effect of changing the input cont

In [21]:
print(retrieved_docs[0].page_content)

1st 25th 50th 75th
Position of Key to Retrieve
40
50
60
70
80
90
100Accuracy
75 Key-Value Pairs (~4K tokens)
1st 35th 70th 105th 140th
Position of Key to Retrieve
40
50
60
70
80
90
100Accuracy
140 Key-Value Pairs (~8K tokens)
1st 50th 100th 150th 200th 250th 300th
Position of Key to Retrieve
40
50
60
70
80
90
100Accuracy
300 Key-Value Pairs (~16K tokens)
claude-1.3 claude-1.3-100k gpt-3.5-turbo-0613 gpt-3.5-turbo-16k-0613 mpt-30b-instruct longchat-13b-16k
Figure 7: The effect of changing the input context length and the position of relevant information on key-value
retrieval performance. Lower positions are closer to the start of the input context. Although some models show
perfect accuracy on this synthetic task (e.g., Claude-1.3 and Claude-1.3 (100K)), we see again that performance is
often highest when relevant information is occurs at the very start or end of the context, and rapidly degrades when
models must retrieve from the middle of the input context.
placed at the start of the

In [23]:
retrieved_docs = retriever.invoke("How BM25 sometimes outperformed dense retrieval ?")

In [24]:
retrieved_docs

[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../Retrieval-Augmented-Generation.pdf', 'total_pages': 19, 'page': 6, 'page_label': '7'}, page_content='Document 1: his works are considered classics of American\nliterature ... His wartime experiences formed the basis for his novel\n”A Farewell to Arms”(1929) ...\nDocument 2: ... artists of the 1920s ”Lost Generation” expatriate\ncommunity. His debut novel,”The Sun Also Rises”, was published\nin 1926.\nBOS\n”\nTheSunAlso\nR ises\n” is a\nnovel\nby this\nauthor\nof ” A\nFarewellto\nArms\n”\nDoc 1\nDoc 2\nDoc 3\nDoc 4\nDoc 5\nFigure 2: RAG-Token document posterior p(zi|x,yi,y−i) for each generated token for input “Hem-\ningway

In [25]:
print(retrieved_docs[0].page_content)

Document 1: his works are considered classics of American
literature ... His wartime experiences formed the basis for his novel
”A Farewell to Arms”(1929) ...
Document 2: ... artists of the 1920s ”Lost Generation” expatriate
community. His debut novel,”The Sun Also Rises”, was published
in 1926.
BOS
”
TheSunAlso
R ises
” is a
novel
by this
author
of ” A
Farewellto
Arms
”
Doc 1
Doc 2
Doc 3
Doc 4
Doc 5
Figure 2: RAG-Token document posterior p(zi|x,yi,y−i) for each generated token for input “Hem-
ingway" for Jeopardy generation with 5 retrieved documents. The posterior for document 1 is high
when generating “A Farewell to Arms" and for document 2 when generating “The Sun Also Rises".
Table 3: Examples from generation tasks. RAG models generate more speciﬁc and factually accurate
responses. ‘?’ indicates factually incorrect responses, * indicates partially correct responses.
Task Input Model Generation
MS-
MARCO
deﬁne middle
ear
BART ?The middle ear is the part of the ear between the middl

In [26]:
print(retrieved_docs[1].page_content)

5 10 20 30 40 50
Number of Retrieved Docs
50
60
70
80
90Metric
claude-1.3
claude-1.3-100k
gpt-3.5-turbo-0613
gpt-3.5-turbo-16k-0613
mpt-30b-instruct
longchat-13b-16k
contriever recall
Figure 11: Retriever recall and model performance as a
function of the number of retrieved documents. Model
performance saturates long before retriever recall, indi-
cating that the models have difficulty making use of the
extra retrieved documents.
domain QA results. We see that reader model
performance saturates long before retriever per-
formance saturates, indicating that readers are not
effectively using the extra context. Using more
than 20 retrieved documents only marginally im-
proves reader performance ( ∼1.5% for GPT-3.5-
Turbo and∼1% for Claude-1.3), while significantly
increasing the input context length (and thus la-
tency and cost). These results, coupled with the
observation that models are often better at retriev-
ing and using information at the start or end of
the input contexts, suggest

In [27]:
print(len(retrieved_docs[1].page_content))

4248


In [28]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [29]:
store1 = InMemoryStore()

In [30]:
vectorstore1 = Chroma(
    collection_name="full_documents", embedding_function=embeddings
)

In [31]:
retriever1 = ParentDocumentRetriever(
    vectorstore=vectorstore1,
    docstore=store1,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [32]:
retriever1.add_documents(docs)

In [33]:

len(list(store1.yield_keys()))

177

In [34]:
retrieved_docs1 = retriever1.invoke("How BM25 sometimes outperformed dense retrieval ?")

In [35]:
retrieved_docs1

[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../Retrieval-Augmented-Generation.pdf', 'total_pages': 19, 'page': 6, 'page_label': '7'}, page_content='To assess the effectiveness of the retrieval mechanism, we run ablations where we freeze the retriever\nduring training. As shown in Table 6, learned retrieval improves results for all tasks.\nWe compare RAG’s dense retriever to a word overlap-based BM25 retriever [53]. Here, we replace\nRAG’s retriever with a ﬁxed BM25 system, and use BM25 retrieval scores as logits when calculating\np(z|x). Table 6 shows the results. For FEVER, BM25 performs best, perhaps since FEVER claims are\nheavily entity-centric and thus well-suited

In [37]:
print(retrieved_docs1[0].page_content)

To assess the effectiveness of the retrieval mechanism, we run ablations where we freeze the retriever
during training. As shown in Table 6, learned retrieval improves results for all tasks.
We compare RAG’s dense retriever to a word overlap-based BM25 retriever [53]. Here, we replace
RAG’s retriever with a ﬁxed BM25 system, and use BM25 retrieval scores as logits when calculating
p(z|x). Table 6 shows the results. For FEVER, BM25 performs best, perhaps since FEVER claims are
heavily entity-centric and thus well-suited for word overlap-based retrieval. Differentiable retrieval
improves results on all other tasks, especially for Open-Domain QA, where it is crucial.
Index hot-swapping An advantage of non-parametric memory models like RAG is that knowledge
can be easily updated at test time. Parametric-only models like T5 or BART need further training to
update their behavior as the world changes. To demonstrate, we build an index using the DrQA [5]


In [38]:
print(len(retrieved_docs1[0].page_content))

960


In [39]:
print(retrieved_docs1[1].page_content)

models with longer input contexts is a trade-off—
providing the language model with more informa-
tion may help it perform the downstream task, but
it also increases the amount of content that the
model must reason over, potentially decreasing ac-
curacy. To better understand this trade-off in prac-
tice, we perform a case study with retriever-reader
models on open-domain question answering (§5).
In contrast to our controlled multi-document QA
task, where the context always contains exactly
one document that answers the question, none or
many of the top k documents may contain the an-
swer in the open-domain QA setting. When re-
trieving from Wikipedia to answer queries from
NaturalQuestions-Open, we find that model perfor-
mance saturates long before retriever recall satu-
rates, indicating that current models fail to effec-
tively use additional retrieved documents—using
50 documents instead of 20 retrieved documents
only marginally improves performance (∼1.5% for


In [40]:
print(len(retrieved_docs1[1].page_content))

980


#### **Data Generation**

In [41]:
from langchain.chains import RetrievalQA

In [45]:
chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)
chain1 = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever1)

In [46]:
query = "How BM25 sometimes outperformed dense retrieval ?"

response = chain.invoke(query)
response

{'query': 'How BM25 sometimes outperformed dense retrieval ?',
 'result': "BM25 sometimes outperformed dense retrieval because FEVER claims are heavily entity-centric, making them well-suited for word overlap-based retrieval methods like BM25. In scenarios where the claims are focused on specific entities, BM25's approach of matching words directly can yield better results compared to dense retrieval methods, which may rely on more complex representations that do not capture the exact matches as effectively."}

In [47]:
response['result']

"BM25 sometimes outperformed dense retrieval because FEVER claims are heavily entity-centric, making them well-suited for word overlap-based retrieval methods like BM25. In scenarios where the claims are focused on specific entities, BM25's approach of matching words directly can yield better results compared to dense retrieval methods, which may rely on more complex representations that do not capture the exact matches as effectively."

"BM25 sometimes outperformed dense retrieval because FEVER claims are heavily entity-centric, making them well-suited for word overlap-based retrieval methods like BM25. In scenarios where the claims are focused on specific entities, BM25's approach of matching words directly can yield better results compared to dense retrieval methods, which may rely on more complex representations that do not capture the exact matches as effectively."

In [48]:
query = "How BM25 sometimes outperformed dense retrieval ?"

response1 = chain1.invoke(query)
response1

{'query': 'How BM25 sometimes outperformed dense retrieval ?',
 'result': 'BM25 sometimes outperformed dense retrieval, particularly in the case of the FEVER task, because FEVER claims are heavily entity-centric. This means that the word overlap-based retrieval method of BM25 is well-suited for retrieving relevant information based on exact matches of words and entities, which can be more effective in this specific context compared to the learned retrieval mechanisms of dense retrieval.'}

In [49]:
response1['result']

'BM25 sometimes outperformed dense retrieval, particularly in the case of the FEVER task, because FEVER claims are heavily entity-centric. This means that the word overlap-based retrieval method of BM25 is well-suited for retrieving relevant information based on exact matches of words and entities, which can be more effective in this specific context compared to the learned retrieval mechanisms of dense retrieval.'

'BM25 sometimes outperformed dense retrieval, particularly in the case of the FEVER task, because FEVER claims are heavily entity-centric. This means that the word overlap-based retrieval method of BM25 is well-suited for retrieving relevant information based on exact matches of words and entities, which can be more effective in this specific context compared to the learned retrieval mechanisms of dense retrieval.'